# LAMA Inpainting Test - Lane Detection

This notebook demonstrates using the LAMA (Large Mask Inpainting) model for lane detection inpainting.
It processes images with pre-generated lane masks and performs inpainting to remove lane markings.

## Requirements
- Images in `../data/sample_images/`
- Corresponding masks in `../data/sample_masks/`
- LAMA model weights in `../models/big-lama/`

## Process
1. Load LAMA model
2. Load image and corresponding lane mask
3. Dilate mask for better coverage
4. Run inpainting inference
5. Save results

In [ ]:
import os
import sys
import torch
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from omegaconf import OmegaConf
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Add parent directory to path for saicinpainting imports
sys.path.insert(0, os.path.abspath('..'))
from saicinpainting.training.trainers import load_checkpoint
from saicinpainting.evaluation.data import pad_img_to_modulo

# Paths (relative to notebook location)
model_dir = "../models/big-lama"
config_path = os.path.join(model_dir, "config.yaml")
checkpoint_path = os.path.join(model_dir, "models/best.ckpt")

# Data directories
img_dir = "../data/sample_images"
mask_dir = "../data/sample_masks"
out_dir = "../data/output"
os.makedirs(out_dir, exist_ok=True)

# Temporary experiment and tensorboard dirs
exp_dir = "../temp/experiments"
tb_dir = "../temp/tb_logs"
os.makedirs(exp_dir, exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# LOAD CONFIG & MODEL
train_config = OmegaConf.load(config_path)
train_config.location.out_root_dir = exp_dir
train_config.location.tb_dir = tb_dir
train_config.visualizer.outdir = os.path.join(exp_dir, "samples")
train_config.data.data_root_dir = "../data"

# Note: This path may need adjustment based on your system
# You can download the weights from: http://sceneparsing.csail.mit.edu/model/pytorch/ade20k-resnet50dilated-ppm_deepsup/
train_config.losses.resnet_pl.weights_path = os.path.expanduser("~/.cache/torch/ade20k/ade20k-resnet50dilated-ppm_deepsup")

model = load_checkpoint(train_config, checkpoint_path, map_location=device, strict=False)
model.eval()
model.to(device)
print("Loaded model checkpoint")

In [ ]:
# IMAGE TRANSFORMS
to_tensor = transforms.ToTensor()
to_pil = transforms.ToPILImage()

# INPAINTING LOOP
for fname in tqdm(os.listdir(img_dir), desc="Inpainting images"):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(img_dir, fname)
    img_base = os.path.splitext(fname)[0]
    mask_fname = f"{img_base}_full.png"
    mask_path = os.path.join(mask_dir, mask_fname)

    if not os.path.exists(mask_path):
        print(f"Mask not found for {fname}, skipping")
        continue

    # Load image and mask
    image = np.array(Image.open(img_path).convert("RGB"))

    # Load and dilate the mask for better coverage
    mask_np = np.array(Image.open(mask_path).convert("L"))
    kernel = np.ones((7, 7), np.uint8)
    dilated_mask = cv2.dilate(mask_np, kernel, iterations=3)
    mask = dilated_mask

    print(f"{fname}: mask min={mask.min()}, max={mask.max()}")

    # Convert to tensor and normalize
    image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
    mask = torch.from_numpy(mask).unsqueeze(0).float() / 255.0

    print(f"Mask unique values: {torch.unique(mask)}")

    # Pad both to multiple of 8
    image = pad_img_to_modulo(image, 8)
    mask = pad_img_to_modulo(mask, 8)

    if isinstance(image, np.ndarray):
        image = torch.from_numpy(image).float()
    if isinstance(mask, np.ndarray):
        mask = torch.from_numpy(mask).float()

    # Visualize original, mask, and masked area
    masked_vis = image.clone()
    masked_vis[:, mask[0] > 0] = 1.0  
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Original")
    plt.imshow(image.permute(1, 2, 0))
    plt.axis('off')
    plt.subplot(1, 3, 2)
    plt.title("Mask")
    plt.imshow(mask[0], cmap='gray')
    plt.axis('off')
    plt.subplot(1, 3, 3)
    plt.title("Masked region")
    plt.imshow(masked_vis.permute(1, 2, 0))
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{img_base}_3vis.png"))
    plt.show()

    # Preparing batch and moving to device
    batch = {
        "image": image.unsqueeze(0).to(device),
        "mask": mask.unsqueeze(0).to(device)
    }

    # Run inference
    with torch.no_grad():
        result = model(batch)
        inpainted = result['inpainted'][0].cpu()

    # Save inpainted image
    out_path = os.path.join(out_dir, fname)
    to_pil(inpainted).save(out_path)

print(f"Inpainting complete! Results saved in: {out_dir}")